# Peptide tokenization and GO-term prediction

Code behind the analyses in *"Vocabulary recurrence, rather than biological grounding, explains
tokenization transfer in protein function prediction"* (Adeyina and Mohl; manuscript in preparation).

The notebook compares four ways of representing a protein, tryptic peptides, overlapping 3-mers,
peptide topics and amino-acid composition, for predicting six Gene Ontology terms. ESM-2 embeddings
and homology transfer serve as reference points. All seeds are fixed at 42, and every stage
checkpoints to disk, so a timed-out session resumes rather than restarts.

Tested on Google Colab (Python 3.12, NVIDIA T4 GPU). Outside Colab, set the environment variables
`TOKENIZATION_ROOT` and `TOKENIZATION_WORK` and install MMseqs2 yourself (see the README).

### How the test set is built

The notebook first builds an all-versus-all homology graph and checks whether a split that keeps
whole connected components apart is possible (§4b). On this corpus it is not, so the test set comes
from MMseqs2 clustering at 30% identity and 80% coverage (§4c). Section 4d then measures the
identity that actually remains between test and training proteins, rather than assuming the
clustering removed it.

### Run order

| § | Stage | Output |
|---|---|---|
| 0–3 | environment, download, parse, all-vs-all search | corpus, alignments |
| 4–4d | homology graph, split attempt, fallback, verification | `test_mask`, identity distribution |
| 5–7 | tokenization, representations, topics | feature matrices |
| 8–8e | evaluation, mechanism, threshold sensitivity, other classifiers, per-class scores | Tables 3–6, S1–S3 |
| 9–10 | stratification, frequency, purity | identity-stratified tables, artefact analysis |
| 11–12 | ESM-2, homology transfer | reference arms |
| 13–15 | summary, figures, manifest | all tables and figures |
| 16 | accession list and split assignment | `corpus_accessions_and_split.tsv` |

## 0. Environment and configuration

In [ ]:
!pip install -q gensim scipy scikit-learn pandas matplotlib

try:                                    # Colab: keep outputs on Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:                     # elsewhere: ROOT comes from TOKENIZATION_ROOT (next cell)
    pass

In [ ]:
import gzip, json, os, pickle, random, re, shutil, subprocess, sys, time
import urllib.parse, urllib.request
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path(os.environ.get("TOKENIZATION_ROOT", "/content/drive/MyDrive/Thesis/v3"))

CFG = dict(
    ROOT=ROOT, RAW_DIR=ROOT/"fasta_raw", CAP_DIR=ROOT/"fasta_capped",
    OUT=ROOT/"results",
    WORK=Path(os.environ.get("TOKENIZATION_WORK", "/content/v3_work")),   # local disk: mmseqs tmp must be executable
                                     # (Drive mounts are not -> "error 13")
    GO_TERMS=["GO:0002933","GO:0004608","GO:0004653",
              "GO:0004713","GO:0015977","GO:0043885"],
    SEED=42, CAP_PER_CLASS=5000, LEN_CAP=5000, LEN_MIN=30,
    MIN_SEQ_ID=0.30, COVERAGE=0.80, SENSITIVITY=7.5, MAX_SEQS=1000,
    TEST_FRACTION=0.20, MIN_TEST_PER_CLASS=30, MAX_DEV_ALLOWED=0.20,
    MISSED_CLEAVAGES=0, MIN_PEP_LEN=6, MAX_PEP_LEN=40, KMER_K=3,
    VECTOR_SIZE=100, WINDOW=5, W2V_EPOCHS=20, MIN_COUNT=5, N_TOPICS=None,
    ESM_MODEL="facebook/esm2_t33_650M_UR50D", ESM_MAXLEN=1022,
    ESM_CHUNK=500, ESM_BATCH=8,
    N_ESTIMATORS=500, N_JOBS=-1,
)
for k in ("RAW_DIR","CAP_DIR","WORK","OUT"): CFG[k].mkdir(parents=True, exist_ok=True)
(CFG["OUT"]/"checkpoints").mkdir(exist_ok=True)
(CFG["OUT"]/"figures").mkdir(exist_ok=True)
(CFG["OUT"]/"tables").mkdir(exist_ok=True)

def log(m): print(f"[{time.strftime('%H:%M:%S')}] {m}", flush=True)
def _ck(n): return CFG["OUT"]/"checkpoints"/f"{n}.pkl"
def save_ck(n,o): pickle.dump(o, open(_ck(n),"wb"), protocol=4)
def load_ck(n):   return pickle.load(open(_ck(n),"rb"))
def have(n):      return _ck(n).exists()
def save_table(df, name):
    df.to_csv(CFG["OUT"]/"tables"/f"{name}.csv")
    return df

log(f"output root: {ROOT}")

## 1. Download

Plain `go:XXXXXXX` queries. **No** reviewed-status filter, no free-text filter.

v1 used a text query plus `reviewed:true`, which produced a SwissProt-only file for GO:0015977
(then dropped entirely by a `tr|`-only header regex, silently removing a class from the reported
table) and an order-of-magnitude undercount for GO:0004713.

Every file is validated against the count the UniProt search API reports, including files already
on disk, and the streaming endpoint falls back to cursor pagination if it terminates early. The
response is sniffed for gzip magic bytes rather than trusting the `compressed` flag.

In [ ]:
STREAM = "https://rest.uniprot.org/uniprotkb/stream"
SEARCH = "https://rest.uniprot.org/uniprotkb/search"
UA = {"User-Agent": "peptide-pipeline/3.0"}

def expected_count(go):
    q = urllib.parse.urlencode({"query": f"(go:{go.split(':')[1]})", "size": "0"})
    with urllib.request.urlopen(urllib.request.Request(f"{SEARCH}?{q}", headers=UA), timeout=120) as r:
        return int(r.headers.get("x-total-results", -1)), r.headers.get("X-UniProt-Release")

def count_fasta(p):
    n = 0
    with open(p, "rb") as fh:
        for line in fh: n += line.startswith(b">")
    return n

def dl_stream(go, dest, attempts=3):
    q = urllib.parse.urlencode({"query": f"(go:{go.split(':')[1]})",
                                "format": "fasta", "compressed": "true"})
    for k in range(attempts):
        try:
            with urllib.request.urlopen(urllib.request.Request(f"{STREAM}?{q}", headers=UA),
                                        timeout=1800) as r:
                blob = r.read()
            if blob[:2] == b"\x1f\x8b": blob = gzip.decompress(blob)
            dest.write_bytes(blob); return True
        except Exception as e:
            log(f"  stream attempt {k+1}/{attempts}: {type(e).__name__}"); time.sleep(5*(k+1))
    return False

def dl_paged(go, dest, size=500):
    url = f"{SEARCH}?" + urllib.parse.urlencode(
        {"query": f"(go:{go.split(':')[1]})", "format": "fasta", "size": str(size)})
    part = dest.with_suffix(".part"); n = 0
    with open(part, "wb") as out:
        while url:
            for k in range(4):
                try:
                    with urllib.request.urlopen(urllib.request.Request(url, headers=UA),
                                                timeout=300) as r:
                        chunk = r.read(); link = r.headers.get("Link", "")
                    break
                except Exception as e:
                    log(f"  page retry {k+1}: {type(e).__name__}"); time.sleep(5*(k+1))
            else: raise RuntimeError(f"{go}: pagination failed")
            out.write(chunk)
            n += chunk.count(b"\n>") + (1 if chunk.startswith(b">") else 0)
            m = re.search(r'<([^>]+)>;\s*rel="next"', link)
            url = m.group(1) if m else None
    part.rename(dest); return True

release = None
for go in CFG["GO_TERMS"]:
    dest = CFG["RAW_DIR"]/f"{go.replace(':','_')}.fasta"
    exp, rel = expected_count(go); release = rel or release
    if dest.exists() and dest.stat().st_size > 0:
        got = count_fasta(dest)
        if exp > 0 and abs(got-exp)/exp < 0.01:
            log(f"{go}: cached and complete ({got:,}/{exp:,})"); continue
        log(f"{go}: cached file INCOMPLETE ({got:,}/{exp:,}) - re-downloading"); dest.unlink()
    log(f"{go}: expecting {exp:,}")
    if not dl_stream(go, dest):
        log(f"{go}: stream failed, using cursor pagination"); dl_paged(go, dest)
    got = count_fasta(dest); ok = exp <= 0 or abs(got-exp)/exp < 0.01
    log(f"{go}: {got:,} sequences, {dest.stat().st_size/1e6:.1f} MB {'OK' if ok else '*** INCOMPLETE ***'}")
    assert ok, f"{go}: got {got:,}, expected {exp:,}"

json.dump({"uniprot_release": release, "downloaded": time.strftime("%Y-%m-%d %H:%M:%S"),
           "counts": {g: count_fasta(CFG["RAW_DIR"]/f"{g.replace(':','_')}.fasta")
                      for g in CFG["GO_TERMS"]}},
          open(CFG["OUT"]/"download_meta.json","w"), indent=2)
log(f"UniProt release: {release}   <-- Methods §2.1")

## 2. Parse, filter, cap

Header regex accepts both `sp|` and `tr|`, capturing description, organism and fragment status
at parse time.

**Multi-label accessions are dropped.** A protein annotated with two of the six terms appears
twice with contradictory labels. An identical input cannot have two correct single-label answers,
so keeping them guarantees a scored error. Logged to `multilabel_dropped.json`.

In [ ]:
HDR = re.compile(
    r"^>(?P<db>sp|tr)\|(?P<acc>[^|]+)\|(?P<entry>\S+)\s+(?P<desc>.*?)"
    r"(?:\s+OS=(?P<org>.*?))?(?:\s+OX=(?P<ox>\d+))?"
    r"(?:\s+GN=(?P<gn>\S+))?(?:\s+PE=(?P<pe>\d))?(?:\s+SV=(?P<sv>\d+))?$")
VALID = set("ACDEFGHIKLMNPQRSTVWYBXZUO")

def iter_fasta(path):
    acc=None; seq=[]; hdr={}
    with open(path, encoding="utf-8", errors="replace") as fh:
        for line in fh:
            line=line.rstrip()
            if line.startswith(">"):
                if acc: yield hdr, "".join(seq)
                m=HDR.match(line)
                if not m: acc,seq,hdr=None,[],{}; continue
                hdr=m.groupdict(); acc,seq=hdr["acc"],[]
            elif acc: seq.append(line)
    if acc: yield hdr, "".join(seq)

if have("df"):
    df = load_ck("df"); log("stage 2: cached")
else:
    rng = random.Random(CFG["SEED"]); rows=[]
    for go in CFG["GO_TERMS"]:
        recs=[]
        for hdr, seq in iter_fasta(CFG["RAW_DIR"]/f"{go.replace(':','_')}.fasta"):
            if not (CFG["LEN_MIN"] <= len(seq) <= CFG["LEN_CAP"]): continue
            if set(seq) - VALID: continue
            recs.append(dict(accession=hdr["acc"], sequence=seq, label=go,
                             database=hdr["db"], entry_name=hdr["entry"],
                             description=(hdr["desc"] or "").strip(),
                             organism=(hdr["org"] or "").strip(),
                             is_fragment=bool(re.search(r"\(Fragment", hdr["desc"] or "", re.I))))
        n_all=len(recs)
        if len(recs) > CFG["CAP_PER_CLASS"]: recs = rng.sample(recs, CFG["CAP_PER_CLASS"])
        log(f"{go}: {n_all:,} parsed -> {len(recs):,} kept")
        rows += recs
    df = pd.DataFrame(rows)
    dup = df.accession.duplicated(keep=False)
    if dup.any():
        multi = df[dup].groupby("accession").label.apply(lambda s: sorted(set(s)))
        json.dump({k:v for k,v in multi.items()},
                  open(CFG["OUT"]/"multilabel_dropped.json","w"), indent=2)
        log(f"dropped {dup.sum()} rows / {df[dup].accession.nunique()} multi-label accessions:")
        for k,v in multi.items(): log(f"    {k}  {v}")
        df = df[~dup]
    df = df.reset_index(drop=True)
    save_ck("df", df)
    df.drop(columns=["sequence"]).to_csv(CFG["OUT"]/"corpus_index.csv", index=False)

with open(CFG["CAP_DIR"]/"corpus.fasta","w") as fh:
    for a,s in zip(df.accession, df.sequence): fh.write(f">{a}\n{s}\n")

# --- TABLE 1 : corpus composition -----------------------------------------
L = df.sequence.str.len()
T1 = (df.assign(L=L).groupby("label")
        .agg(n=("accession","size"), median_len=("L","median"),
             n_fragment=("is_fragment","sum"), pct_fragment=("is_fragment","mean"),
             n_sp=("database", lambda s:(s=="sp").sum()))
        .assign(pct_truncated_1022=df.assign(L=L).groupby("label").L.apply(lambda s:(s>1022).mean())))
save_table(T1.round(4), "T1_corpus_composition")
log(f"corpus: {len(df):,} proteins | database mix {df.database.value_counts().to_dict()}")
log(f"fragments {df.is_fragment.sum():,} ({df.is_fragment.mean():.1%}) | "
    f">1022 aa {(L>1022).sum():,} ({(L>1022).mean():.1%})")
print(T1.round(3).to_string())

## 3. All-versus-all homology search

`-c 0.80 --cov-mode 0` requires **bidirectional** coverage, so an edge means two sequences align
over most of *both* lengths rather than sharing a domain in otherwise unrelated proteins.

`tmp` and the raw output go to local disk: Drive mounts are non-executable, and MMseqs2 writes
then executes a shell script in `tmp`. That is the `error 13` (EACCES) failure. Budget about an
hour.

In [ ]:
if not shutil.which("mmseqs"):
    log("installing mmseqs2 ...")
    !wget -q https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz -O /tmp/m.tar.gz
    !tar xzf /tmp/m.tar.gz -C /tmp && cp /tmp/mmseqs/bin/mmseqs /usr/local/bin/ && chmod +x /usr/local/bin/mmseqs
MMSEQS_VERSION = subprocess.run(["mmseqs","version"], capture_output=True, text=True).stdout.strip()
log(f"mmseqs version: {MMSEQS_VERSION}")

ALL = CFG["OUT"]/"allvall.m8"
if ALL.exists() and ALL.stat().st_size > 0:
    log(f"stage 3: cached ({ALL.stat().st_size/1e6:.0f} MB)")
else:
    tmp = CFG["WORK"]/"mmseqs_tmp"; shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir(parents=True)
    local = CFG["WORK"]/"allvall.m8"; fa = str(CFG["CAP_DIR"]/"corpus.fasta")
    log("all-vs-all search (long stage) ...")
    r = subprocess.run(["mmseqs","easy-search", fa, fa, str(local), str(tmp),
        "-s", str(CFG["SENSITIVITY"]), "-e","1e-3", "--max-seqs", str(CFG["MAX_SEQS"]),
        "-c", str(CFG["COVERAGE"]), "--cov-mode","0", "--threads", str(os.cpu_count() or 4),
        "--format-output","query,target,fident,alnlen,qcov,tcov,evalue,bits"],
        capture_output=True, text=True)
    if r.returncode != 0: print(r.stderr[-3000:]); raise RuntimeError("easy-search failed")
    shutil.copy(local, ALL); log(f"done: {ALL.stat().st_size/1e6:.0f} MB")

M8 = dict(sep="\t", header=None,
          names=["q","t","fident","alnlen","qcov","tcov","evalue","bits"])

## 4. Homology graph and component structure

Edges join sequences at ≥`MIN_SEQ_ID` identity with ≥`COVERAGE` bidirectional coverage.
Connected components of that graph are **transitively closed**: if A relates to B and B to C, all
three sit in one group and necessarily fall on the same side of any group-aware partition.

This is the structurally correct grouping for a homology-disjoint split. Whether it is *usable*
is an empirical question, answered in §4b.

In [ ]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import StratifiedGroupKFold

if have("components"):
    comp, ncomp = load_ck("components"); log("stage 4: cached")
else:
    idx = {a:i for i,a in enumerate(df.accession)}; n = len(df)
    qs, ts = [], []
    for ch in pd.read_csv(ALL, chunksize=2_000_000, **M8):
        c = ch[(ch.fident >= CFG["MIN_SEQ_ID"]) & (ch.qcov >= CFG["COVERAGE"]) &
               (ch.tcov >= CFG["COVERAGE"]) & (ch.q != ch.t)]
        qs.append(c.q.map(idx).to_numpy()); ts.append(c.t.map(idx).to_numpy())
    qi = np.concatenate(qs); ti = np.concatenate(ts)
    ok = ~(pd.isna(qi) | pd.isna(ti)); qi, ti = qi[ok].astype(int), ti[ok].astype(int)
    log(f"edges at >={CFG['MIN_SEQ_ID']:.0%} id / >={CFG['COVERAGE']:.0%} cov both ways: {len(qi):,}")
    A = coo_matrix((np.ones(len(qi), np.int8), (qi, ti)), shape=(n,n))
    ncomp, comp = connected_components(A, directed=False)
    save_ck("components", (comp, ncomp))

sizes = pd.Series(comp).value_counts()
log(f"components: {ncomp:,} | largest {sizes.iloc[0]:,} ({sizes.iloc[0]/len(df):.1%}) "
    f"| top10 {list(sizes.head(10))}")

# --- TABLE 2 : component structure by class -------------------------------
cs = pd.DataFrame({"comp": comp, "label": df.label.to_numpy()})
T2 = cs.groupby("label").comp.agg(n_components="nunique", n_proteins="size")
T2["largest_comp_share"] = cs.groupby("label").comp.apply(lambda s: s.value_counts().iloc[0]/len(s))
save_table(T2.round(4), "T2_component_structure")
print(T2.round(3).to_string())

### 4b. The component-disjoint split is unattainable

A connected component is **atomic** under any group-aware splitter: it goes wholly to training or
wholly to test. Where a class is concentrated in one component, no stratified allocation exists.

This cell evaluates all folds and reports the outcome rather than raising. The result is a
finding reported in the manuscript, not an error to be worked around.

In [ ]:
def fold_report(groups, tag):
    sgkf = StratifiedGroupKFold(n_splits=max(2, round(1/CFG["TEST_FRACTION"])),
                                shuffle=True, random_state=CFG["SEED"])
    out = []
    for i,(tr,te) in enumerate(sgkf.split(np.zeros(len(df)), df.label.to_numpy(), groups=groups)):
        m = np.zeros(len(df), bool); m[te] = True
        ct = pd.crosstab(df.label, m)
        test_n = ct.get(True, pd.Series(0, index=ct.index))
        fr = test_n/ct.sum(1)
        dev = float(np.abs(fr - CFG["TEST_FRACTION"]).max())
        # A fold is usable only if BOTH hold: every class keeps enough test
        # proteins, AND no class deviates wildly from the target test fraction.
        # Checking min_class alone admits folds that invert the split entirely
        # (e.g. a class at 98% test), which is not a usable partition.
        out.append(dict(fold=i, n_test=int(m.sum()), min_class=int(test_n.min()),
                        max_dev=dev,
                        usable=bool(test_n.min() >= CFG["MIN_TEST_PER_CLASS"]
                                    and dev <= CFG["MAX_DEV_ALLOWED"]),
                        mask=m))
    F = pd.DataFrame(out).drop(columns="mask")
    log(f"--- {tag} ---"); print(F.to_string(index=False))
    return out, F

comp_folds, F_comp = fold_report(comp, "component-disjoint (transitively closed)")
save_table(F_comp, "T3a_component_folds")

if not F_comp.usable.any():
    log(f"RESULT: no component-disjoint fold gives every class >= "
        f"{CFG['MIN_TEST_PER_CLASS']} test proteins within "
        f"{CFG['MAX_DEV_ALLOWED']:.0%} of the target test fraction.")
    log("        Homology-disjoint splitting and class balance are mutually")
    log("        unsatisfiable on this corpus. Reported as a methodological result.")
else:
    log("RESULT: a usable component-disjoint fold exists; prefer it over the §4c fallback.")

### 4c. Adopted split: cluster-disjoint

Falling back to the standard approach, MMseqs2 `easy-cluster` at the same identity and coverage
thresholds. Of the available folds we take the one minimising the maximum per-class deviation
from the target test fraction, subject to every class retaining at least `MIN_TEST_PER_CLASS`
test proteins. Fold selection is a stated choice, not a silent one.

In [ ]:
clu_tsv = CFG["OUT"]/"clusters.tsv"
if not clu_tsv.exists():
    tmp = CFG["WORK"]/"clu_tmp"; shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir(parents=True)
    pre = CFG["WORK"]/"clu"
    r = subprocess.run(["mmseqs","easy-cluster", str(CFG["CAP_DIR"]/"corpus.fasta"),
        str(pre), str(tmp), "--min-seq-id", str(CFG["MIN_SEQ_ID"]),
        "-c", str(CFG["COVERAGE"]), "--cov-mode","0", "--threads", str(os.cpu_count() or 4)],
        capture_output=True, text=True)
    if r.returncode != 0: print(r.stderr[-3000:]); raise RuntimeError("easy-cluster failed")
    shutil.copy(f"{pre}_cluster.tsv", clu_tsv)

clu = pd.read_csv(clu_tsv, sep="\t", header=None, names=["rep","member"])
groups = df.accession.map(dict(zip(clu.member, clu.rep))).fillna(df.accession).to_numpy()
n_clusters = pd.Series(groups).nunique()
log(f"MMseqs2 clusters: {n_clusters:,} ({len(df)/n_clusters:.1f}x redundancy)")

clu_folds, F_clu = fold_report(groups, "cluster-disjoint")
save_table(F_clu, "T3b_cluster_folds")

usable = [f for f in clu_folds if f["usable"]] or clu_folds
best = min(usable, key=lambda f: f["max_dev"])
test_mask = best["mask"]; FOLD_USED = best["fold"]
assert not (set(groups[~test_mask]) & set(groups[test_mask])), "cluster leak"

T4 = pd.crosstab(df.label, test_mask); T4.columns = ["train","test"]
T4["test_frac"] = (T4.test/T4.sum(1)).round(3)
save_table(T4, "T4_split_composition")
save_ck("split", dict(test_mask=test_mask, groups=groups, components=comp,
                      fold_used=FOLD_USED, n_clusters=int(n_clusters),
                      strategy="cluster_disjoint"))
log(f"adopted fold {FOLD_USED}: train {(~test_mask).sum():,} | test {test_mask.sum():,}")
print(T4.to_string())

In [ ]:
# Guard for cached matrices: does this run reproduce the split reported in the manuscript?
EXPECTED = dict(fold=3, n_test=3181, n_clusters=2636)
GOT = dict(fold=int(FOLD_USED), n_test=int(test_mask.sum()), n_clusters=int(n_clusters))
if GOT == EXPECTED:
    print("split reproduced (fold 3, 3,181 test proteins, 2,636 clusters): cached matrices are valid")
else:
    log(f"WARNING: split differs from the manuscript. expected {EXPECTED}, got {GOT}. "
        "A newer UniProt release gives a different corpus, so numbers will differ. "
        "Delete results/checkpoints/ before continuing so cached matrices are not reused.")

### 4d. What the split actually controls

**The check v2 lacked.** Greedy clustering forms clusters whose members share an edge with a
cluster *representative*, which constrains member-to-representative similarity but not similarity
between members of different clusters. Cluster-disjointness therefore does not imply pairwise
identity control.

For every test protein we take the maximum identity to any training protein over alignments
covering at least `COVERAGE` of both sequences. This distribution is reported in the manuscript,
and §9 stratifies on it.

In [ ]:
if (CFG["OUT"]/"max_global_ident.npy").exists():
    gi = np.load(CFG["OUT"]/"max_global_ident.npy"); log("stage 4d: cached")
else:
    tr_acc = set(df.accession[~test_mask]); best_hit = {}
    for ch in pd.read_csv(ALL, chunksize=2_000_000, **M8):
        c = ch[(ch.qcov >= CFG["COVERAGE"]) & (ch.tcov >= CFG["COVERAGE"]) & (ch.q != ch.t)]
        c = c[c.t.isin(tr_acc)]
        for q,f in zip(c.q.to_numpy(), c.fident.to_numpy()):
            if f > best_hit.get(q, 0.0): best_hit[q] = f
    gi = df.accession[test_mask].map(best_hit).fillna(0.0).to_numpy()
    np.save(CFG["OUT"]/"max_global_ident.npy", gi)

T5 = pd.Series({f">={t:.0%}": (gi>=t).mean() for t in (0.30,0.40,0.50,0.90)}).to_frame("fraction")
T5["n"] = [(gi>=t).sum() for t in (0.30,0.40,0.50,0.90)]
save_table(T5.round(4), "T5_identity_distribution")
log(f"max identity of TEST to TRAIN (>= {CFG['COVERAGE']:.0%} bidirectional coverage):")
print(pd.Series(gi).describe().round(3).to_string()); print(); print(T5.round(3).to_string())
log("Split is NOT homology-disjoint. Performance is reported against measured identity (§9).")

## 5. Tokenization

Trypsin cleaves C-terminal to K or R except before P: `(?<=[KR])(?!P)`.

In [ ]:
TRYPSIN = re.compile(r"(?<=[KR])(?!P)")

def digest(seq, missed=0, lo=6, hi=40):
    peps = [p for p in TRYPSIN.split(seq) if p]
    out = list(peps)
    for m in range(1, missed+1):
        out += ["".join(peps[i:i+m+1]) for i in range(len(peps)-m)]
    return [p for p in out if lo <= len(p) <= hi]

def kmers(seq, k=3): return [seq[i:i+k] for i in range(len(seq)-k+1)]

assert digest("AAAKPBBBRCCCK",0,1,40) == ["AAAKPBBBR","CCCK"]     # K before P: no cut
assert digest("AAAKBBBK",1,1,40)      == ["AAAK","BBBK","AAAKBBBK"]
assert kmers("ABCDE",3) == ["ABC","BCD","CDE"]

if have("tokens"): tokens = load_ck("tokens"); log("stage 5: cached")
else:
    tokens = {"peptide":[digest(s, CFG["MISSED_CLEAVAGES"], CFG["MIN_PEP_LEN"], CFG["MAX_PEP_LEN"])
                         for s in df.sequence],
              "kmer":[kmers(s, CFG["KMER_K"]) for s in df.sequence]}
    save_ck("tokens", tokens)

vc_pep = pd.Series([p for ps in tokens["peptide"] for p in ps]).value_counts()
VOCAB_STATS = dict(n_tokens=int(vc_pep.sum()), n_unique=int(len(vc_pep)),
                   singleton_frac=float((vc_pep==1).mean()),
                   n_kmer_unique=len({k for ks in tokens["kmer"] for k in ks}))
json.dump(VOCAB_STATS, open(CFG["OUT"]/"vocab_stats.json","w"), indent=2)
log(f"peptides: {VOCAB_STATS['n_tokens']:,} tokens | {VOCAB_STATS['n_unique']:,} unique "
    f"| {VOCAB_STATS['singleton_frac']:.1%} singletons")
log(f"3-mers: {VOCAB_STATS['n_kmer_unique']:,} unique of 8,000 possible")

## 6. Representations, fitted on train only

Word2Vec sees training sequences only; out-of-vocabulary rates on test are measured rather than
avoided. `workers=1` for determinism (gensim is non-deterministic with multiple workers
regardless of seed). Amino-acid composition is the floor: 20 features, no order, no learned
tokens.

In [ ]:
from gensim.models import Word2Vec

def doc_vecs(docs, kv, dim):
    X = np.zeros((len(docs), dim), np.float32); cov = np.zeros(len(docs), np.float32)
    for i,d in enumerate(docs):
        v = [kv[t] for t in d if t in kv]
        if v: X[i] = np.mean(v, axis=0)
        cov[i] = len(v)/max(1, len(d))
    return X, cov

def fit_w2v(docs, train_idx, min_count):
    return Word2Vec([docs[i] for i in train_idx], vector_size=CFG["VECTOR_SIZE"],
                    window=CFG["WINDOW"], min_count=min_count, epochs=CFG["W2V_EPOCHS"],
                    sg=1, workers=1, seed=CFG["SEED"]).wv

if have("matrices"): mats, extra = load_ck("matrices"); log("stage 6: cached")
else:
    mats, extra = {}, {}
    AAs = "ACDEFGHIKLMNPQRSTVWY"
    X_aac = np.zeros((len(df), 20), np.float32)
    for i,s in enumerate(df.sequence):
        L = max(1, len(s))
        for j,a in enumerate(AAs): X_aac[i,j] = s.count(a)/L
    mats["aa_composition"] = (X_aac[~test_mask], X_aac[test_mask])

    tr_idx = np.where(~test_mask)[0]; te_idx = np.where(test_mask)[0]
    for arm in ("kmer","peptide"):
        docs = tokens[arm]
        kv = fit_w2v(docs, tr_idx, CFG["MIN_COUNT"])
        X, cov = doc_vecs(docs, kv, CFG["VECTOR_SIZE"])
        mats[arm] = (X[~test_mask], X[test_mask])
        seen = sum(1 for i in te_idx for t in docs[i] if t in kv)
        tot  = sum(len(docs[i]) for i in te_idx)
        extra[f"{arm}_coverage"] = cov
        extra[f"{arm}_oov"] = 1 - seen/max(1,tot)
        extra[f"{arm}_zero"] = int((np.abs(X[test_mask]).sum(1)==0).sum())
        extra[f"{arm}_vocab"] = len(kv)
        log(f"{arm}: vocab {len(kv):,} | token OOV {extra[f'{arm}_oov']:.1%} "
            f"| zero-vector test proteins {extra[f'{arm}_zero']:,}")
        if arm == "peptide": extra["peptide_kv"] = {w: kv[w] for w in kv.index_to_key}
    save_ck("matrices", (mats, extra))

## 7. Peptide topics

Topics are clusters of the **peptide vocabulary**; each protein is a count vector over them.
v1 trained a second Word2Vec on BERTopic *label strings*, making every protein in a topic
identical, which is a lookup table rather than a representation.

One extra column holds out-of-vocabulary peptides, so nothing is discarded. v1's topic arm
silently dropped 28% of proteins (the BERTopic `-1` bin) and compared the remainder head-to-head
against a full-corpus arm.

In [ ]:
from sklearn.cluster import MiniBatchKMeans

if have("topics"):
    topics = load_ck("topics"); mats, extra = load_ck("matrices"); log("stage 7: cached")
else:
    kv = extra["peptide_kv"]; vocab = list(kv); V = np.vstack([kv[w] for w in vocab])
    k = CFG["N_TOPICS"] or int(round(np.sqrt(len(vocab))))
    log(f"N_TOPICS = {k} ({'configured' if CFG['N_TOPICS'] else 'sqrt(vocab) heuristic'}) "
        f"over a {len(vocab):,}-peptide vocabulary")
    km = MiniBatchKMeans(n_clusters=k, random_state=CFG["SEED"], n_init=10, batch_size=4096).fit(V)
    topic_of = dict(zip(vocab, km.labels_))
    X = np.zeros((len(df), k+1), np.float32)
    for i,doc in enumerate(tokens["peptide"]):
        for t in doc: X[i, topic_of.get(t, k)] += 1
    mats["topic"] = (X[~test_mask], X[test_mask])
    topics = {"X": X, "topic_of": topic_of, "k": k}
    save_ck("topics", topics); save_ck("matrices", (mats, extra))

oov_share = topics["X"][test_mask][:,-1].sum()/topics["X"][test_mask].sum()
log(f"topic matrix {topics['X'].shape} (last column = OOV) | "
    f"test peptide mass in OOV column: {oov_share:.1%}")

## 8. Evaluation

Every arm on the identical frozen test set.

**All macro averages are computed over the fixed six-label set**, not over the labels appearing in
a given arm's predictions. Without this, an arm that never predicts a class averages over five
while another averages over six, and the arms are not comparable. This also prevents the
`ValueError` that arises when an arm's predictions omit a class.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (f1_score, balanced_accuracy_score, accuracy_score,
                             precision_score, recall_score, average_precision_score,
                             roc_auc_score, confusion_matrix, classification_report)

le = LabelEncoder(); y = le.fit_transform(df.label)
y_train, y_test = y[~test_mask], y[test_mask]
ALL_LABELS = np.arange(len(le.classes_))
Yb = label_binarize(y_test, classes=ALL_LABELS)

rows, preds, reports, models, probas = [], {}, {}, {}, {}

def run_arm(name, Xtr, Xte):
    clf = RandomForestClassifier(n_estimators=CFG["N_ESTIMATORS"], random_state=CFG["SEED"],
                                 n_jobs=CFG["N_JOBS"],
                                 class_weight="balanced_subsample").fit(Xtr, y_train)
    p = clf.predict(Xte); pr = clf.predict_proba(Xte)
    preds[name] = p; probas[name] = pr; models[name] = clf
    reports[name] = classification_report(y_test, p, labels=ALL_LABELS,
                                          target_names=le.classes_, output_dict=True,
                                          zero_division=0)
    rows.append(dict(arm=name, n_features=Xtr.shape[1],
                     accuracy=accuracy_score(y_test,p),
                     balanced_accuracy=balanced_accuracy_score(y_test,p),
                     macro_precision=precision_score(y_test,p,labels=ALL_LABELS,average="macro",zero_division=0),
                     macro_recall=recall_score(y_test,p,labels=ALL_LABELS,average="macro",zero_division=0),
                     macro_f1=f1_score(y_test,p,labels=ALL_LABELS,average="macro",zero_division=0),
                     weighted_f1=f1_score(y_test,p,labels=ALL_LABELS,average="weighted",zero_division=0),
                     macro_aupr=average_precision_score(Yb, pr, average="macro"),
                     roc_auc=roc_auc_score(Yb, pr, average="macro", multi_class="ovr")))
    np.save(CFG["OUT"]/f"cm_{name}.npy", confusion_matrix(y_test, p, labels=ALL_LABELS))
    log(f"{name:<18} macro-F1 {rows[-1]['macro_f1']:.4f}  bal-acc "
        f"{rows[-1]['balanced_accuracy']:.4f}  AUPR {rows[-1]['macro_aupr']:.4f}")
    return clf

for s in ("most_frequent","stratified"):
    d = DummyClassifier(strategy=s, random_state=CFG["SEED"]).fit(np.zeros((len(y_train),1)), y_train)
    p = d.predict(np.zeros((len(y_test),1))); preds[f"baseline_{s}"] = p
    rows.append(dict(arm=f"baseline_{s}", n_features=np.nan,
                     accuracy=accuracy_score(y_test,p),
                     balanced_accuracy=balanced_accuracy_score(y_test,p),
                     macro_f1=f1_score(y_test,p,labels=ALL_LABELS,average="macro",zero_division=0)))

for name in ("aa_composition","kmer","peptide","topic"):
    run_arm(name, *mats[name])

save_ck("preds", (preds, le, y_test))
print(); print(pd.DataFrame(rows).round(4).to_string(index=False))

### 8b. Mechanism: the two populations

The central result. If the peptide arm produces zero vectors, those proteins are **identical
inputs** and receive one constant prediction. Conditioning per-class accuracy on coverage
separates the two populations and shows there is no intermediate regime.

This produces Table 4 of the manuscript.

In [ ]:
Xte_p = mats["peptide"][1]
zr = (np.abs(Xte_p).sum(1) == 0)
p_pep = preds["peptide"]

log(f"zero-vector test proteins: {zr.sum():,} ({zr.mean():.1%})")
if zr.sum():
    log(f"unique rows among them  : {len(np.unique(Xte_p[zr], axis=0))}")
    log(f"predictions             : {pd.Series(le.inverse_transform(p_pep[zr])).value_counts().to_dict()}")
    log(f"accuracy | zero-vector  : {(p_pep[zr]==y_test[zr]).mean():.3f}  "
        f"(= prevalence of the constant class in this group)")
    log(f"accuracy | non-zero     : {(p_pep[~zr]==y_test[~zr]).mean():.3f}")

# --- TABLE 6 : per-class accuracy conditioned on coverage -----------------
T6 = pd.DataFrame([dict(
        cls=le.classes_[c], n=int((y_test==c).sum()), n_zero=int(((y_test==c)&zr).sum()),
        acc_all=float((p_pep[y_test==c]==c).mean()),
        acc_covered=float((p_pep[(y_test==c)&~zr]==c).mean()) if ((y_test==c)&~zr).sum() else np.nan,
        acc_zero=float((p_pep[(y_test==c)&zr]==c).mean()) if ((y_test==c)&zr).sum() else np.nan)
    for c in ALL_LABELS]).set_index("cls")
save_table(T6.round(4), "T6_coverage_conditioned")
print(); print(T6.round(3).to_string())

# accuracy by coverage bin
cov = extra["peptide_coverage"][test_mask]
b = pd.cut(cov, [-.01,.001,.25,.50,.75,1.01], labels=["0","<25%","25-50%","50-75%",">75%"])
T7 = (pd.DataFrame({"bin": b, "correct": p_pep==y_test})
        .groupby("bin", observed=True).correct.agg(n="size", accuracy="mean"))
save_table(T7.round(4), "T7_accuracy_by_coverage")
print(); print(T7.round(3).to_string())

### 8c. Vocabulary threshold sensitivity

`MIN_COUNT=5` discards most of the peptide vocabulary, so a reviewer will reasonably ask whether
the failure is an artefact of that choice. This refits the arm at three thresholds and answers
with a number rather than an argument.

Note that token OOV and abandoned-protein count are **different quantities** and respond
differently to the threshold.

In [ ]:
if have("min_count_sensitivity"):
    T8 = load_ck("min_count_sensitivity"); log("stage 8c: cached")
else:
    docs = tokens["peptide"]; tr_idx = np.where(~test_mask)[0]; te_idx = np.where(test_mask)[0]
    out = []
    for mc in (1, 2, 5):
        kv = fit_w2v(docs, tr_idx, mc)
        X, _ = doc_vecs(docs, kv, CFG["VECTOR_SIZE"])
        seen = sum(1 for i in te_idx for t in docs[i] if t in kv)
        tot  = sum(len(docs[i]) for i in te_idx)
        zero = int((np.abs(X[test_mask]).sum(1)==0).sum())
        clf = RandomForestClassifier(n_estimators=CFG["N_ESTIMATORS"], random_state=CFG["SEED"],
                                     n_jobs=CFG["N_JOBS"],
                                     class_weight="balanced_subsample").fit(X[~test_mask], y_train)
        pp = clf.predict(X[test_mask])
        out.append(dict(min_count=mc, vocab=len(kv), token_oov=1-seen/tot,
                        zero_vec=zero, zero_pct=zero/test_mask.sum(),
                        macro_f1=f1_score(y_test, pp, labels=ALL_LABELS,
                                          average="macro", zero_division=0)))
        log(f"min_count={mc}: vocab {len(kv):,} | OOV {out[-1]['token_oov']:.1%} "
            f"| zero {zero:,} | macro-F1 {out[-1]['macro_f1']:.4f}")
    T8 = pd.DataFrame(out); save_ck("min_count_sensitivity", T8)

save_table(T8.round(4), "T8_min_count_sensitivity")
print(); print(T8.round(4).to_string(index=False))

### 8d. Do the comparisons hold under other classifiers?

Logistic regression and gradient boosting on the same frozen split, with the random-forest values taken from section 8. The abstention mechanism is also checked per classifier.

In [ ]:
# ---------------------------------------------------------------------------
# 8d. Classifier robustness: do the arm comparisons hold under other models?
#
# Needs objects already defined by earlier cells: mats, y_train, y_test, Yb,
# ALL_LABELS, le, rows, preds, CFG, log, save_table.
# Nothing here changes any existing result. Writes T13_classifier_robustness (manuscript
# Table 6 and Table S1) and T14_abstention_by_classifier (Table S2).
# ---------------------------------------------------------------------------
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.utils.class_weight import compute_sample_weight

ARMS = ("aa_composition", "kmer", "peptide", "topic")


def make_models():
    return {
        # linear model: the opposite extreme from a tree ensemble
        "logistic_regression": make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=2000, class_weight="balanced",
                               random_state=CFG["SEED"])),
        # boosted trees: a stronger tree-based alternative to the forest
        "gradient_boosting": HistGradientBoostingClassifier(
            max_iter=200, early_stopping=False, random_state=CFG["SEED"]),
    }


def fit_predict(name, model, Xtr, Xte):
    if name == "gradient_boosting":     # same balancing intent as the forest
        model.fit(Xtr, y_train, sample_weight=compute_sample_weight("balanced", y_train))
    else:
        model.fit(Xtr, y_train)
    assert np.array_equal(model.classes_, ALL_LABELS), "class order mismatch"
    return model.predict(Xte), model.predict_proba(Xte)


def score(p, pr):
    return dict(
        macro_f1=f1_score(y_test, p, labels=ALL_LABELS, average="macro", zero_division=0),
        balanced_accuracy=balanced_accuracy_score(y_test, p),
        macro_aupr=average_precision_score(Yb, pr, average="macro"),
        roc_auc=roc_auc_score(Yb, pr, average="macro", multi_class="ovr"))


# --- random forest reference, taken from the existing run (not refitted) ----
rob = [dict(classifier="random_forest", arm=r["arm"],
            **{k: r[k] for k in ("macro_f1", "balanced_accuracy", "macro_aupr", "roc_auc")})
       for r in rows if r["arm"] in ARMS]
all_preds = {("random_forest", a): preds[a] for a in ARMS}

# --- the two other classifiers, same frozen split, same metrics -------------
for cname in make_models():
    for arm in ARMS:
        Xtr, Xte = mats[arm]
        p, pr = fit_predict(cname, make_models()[cname], Xtr, Xte)
        all_preds[(cname, arm)] = p
        rob.append(dict(classifier=cname, arm=arm, **score(p, pr)))
        log(f"{cname:<20} {arm:<15} macro-F1 {rob[-1]['macro_f1']:.4f}")

T13 = pd.DataFrame(rob)
save_table(T13.round(4), "T13_classifier_robustness")
print()
print(T13.pivot(index="arm", columns="classifier", values="macro_f1")
         .reindex(list(ARMS)).round(4).to_string())

# --- does the ordering survive? --------------------------------------------
print()
for cname in T13.classifier.unique():
    s = T13[T13.classifier == cname].set_index("arm").macro_f1
    log(f"{cname:<20} kmer {s['kmer']:.4f} | peptide {s['peptide']:.4f} | "
        f"gap (kmer - peptide) {s['kmer'] - s['peptide']:+.4f} | "
        f"ranking {' > '.join(s.sort_values(ascending=False).index)}")

# --- the abstention mechanism, per classifier ------------------------------
zr_ = (np.abs(mats["peptide"][1]).sum(1) == 0)
mech = []
for (cname, arm), p in all_preds.items():
    if arm != "peptide":
        continue
    mech.append(dict(classifier=cname, n_zero_vector=int(zr_.sum()),
                     distinct_predictions_among_zero=int(len(np.unique(p[zr_]))),
                     acc_zero_vector=float((p[zr_] == y_test[zr_]).mean()),
                     acc_covered=float((p[~zr_] == y_test[~zr_]).mean())))
T14 = pd.DataFrame(mech)
save_table(T14.round(4), "T14_abstention_by_classifier")
print(); print(T14.round(3).to_string(index=False))
log("distinct_predictions_among_zero == 1 means all zero-vector proteins got one constant "
    "prediction: abstention is a property of the representation, not of the classifier.")

### 8e. Per-class precision and recall

Supplement Table S3.

In [ ]:
# Per-class precision, recall and F1 for each learned arm (supplement Table S3).
# Uses the per-class reports already stored by run_arm in section 8.
PC_ARMS = ("peptide", "kmer", "topic", "aa_composition")
pc = []
for arm in PC_ARMS:
    rep = (pd.DataFrame(reports[arm]).T
             .loc[list(le.classes_), ["precision", "recall", "f1-score", "support"]])
    rep.insert(0, "class", rep.index)
    rep.insert(0, "arm", arm)
    pc.append(rep.reset_index(drop=True))
S3 = pd.concat(pc, ignore_index=True)
S3[["precision", "recall", "f1-score"]] = S3[["precision", "recall", "f1-score"]].round(4)
S3.to_csv(CFG["OUT"]/"tables"/"S3_per_class_precision_recall.csv", index=False)
print(S3.to_string(index=False))

## 9. Identity-stratified performance, and why it is descriptive only

Accuracy against maximum identity to the nearest training protein. Before reading any gradient
into this, two checks are required:

1. **Class composition per stratum.** If the strata contain different classes, they are not
   comparable populations and a trivial baseline will vary across them.
2. **Within-class stratification.** Holding class fixed removes the composition confound.

Both are computed here. The manuscript reports this analysis as descriptive, because the
strata are confounded and the gradient does not survive within class.

In [ ]:
LABELS_ID = ["no global hit","<30%","30-50%","50-90%",">90%"]
grp = pd.Series(pd.cut(gi, [-.01,.001,.30,.50,.90,1.01], labels=LABELS_ID))

rows_s = []
for name, p in preds.items():
    for g in LABELS_ID:
        m = (grp==g).to_numpy()
        if m.sum() >= 20:
            rows_s.append(dict(arm=name, bin=g, n=int(m.sum()),
                               accuracy=float((p[m]==y_test[m]).mean())))
T9 = pd.DataFrame(rows_s)
save_table(T9, "T9_identity_stratified")
print(T9.pivot(index="bin", columns="arm", values="accuracy").reindex(LABELS_ID).round(3).to_string())

# --- confound check 1: class composition of each stratum ------------------
T10 = pd.crosstab(grp, pd.Series(le.inverse_transform(y_test)), normalize="index").reindex(LABELS_ID)
save_table(T10.round(4), "T10_stratum_class_composition")
print("\nclass composition of each identity stratum:"); print(T10.round(3).to_string())
log("A trivial baseline varying across strata indicates composition differences, not difficulty.")

# --- confound check 2: within-class stratification ------------------------
rows_w = []
for c in ALL_LABELS:
    mc_ = (y_test==c)
    for g in LABELS_ID:
        m = mc_ & (grp==g).to_numpy()
        if m.sum() >= 30:
            for name in ("peptide","kmer","aa_composition"):
                rows_w.append(dict(cls=le.classes_[c], bin=g, n=int(m.sum()), arm=name,
                                   accuracy=float((preds[name][m]==c).mean())))
T11 = pd.DataFrame(rows_w)
save_table(T11, "T11_within_class_stratified")
print("\nwithin-class (composition controlled by construction):")
if len(T11): print(T11.pivot_table(index=["cls","bin"], columns="arm", values="accuracy").round(3).to_string())

## 10. Peptide frequency, database artefacts, and topic purity

Three diagnostics establishing that the peptide arm's apparent successes track database
composition rather than biology:

- **Frequency ranking** dominated by one heavily over-sampled conserved protein
- **Fragment enrichment** among top-ranked peptides relative to the corpus baseline
- **Topic purity**, showing clusters are family detectors rather than motif detectors

Purity is partly determined by construction, since Word2Vec embeds peptide co-occurrence within
proteins, so same-protein peptides co-locate and are clustered together. That circularity is the
finding: any topic model over peptide embeddings recovers families if peptide identity carries
family membership and little else.

In [ ]:
acc2meta = df.set_index("accession")[["label","description","is_fragment"]].to_dict("index")

if (CFG["OUT"]/"peptide_frequencies.csv").exists():
    freq = pd.read_csv(CFG["OUT"]/"peptide_frequencies.csv"); log("stage 10: cached")
else:
    rows_f = [(p, df.accession.iloc[i]) for i,doc in enumerate(tokens["peptide"]) for p in set(doc)]
    PF = pd.DataFrame(rows_f, columns=["peptide","accession"])
    freq = (PF.groupby("peptide")
              .agg(n_proteins=("accession","size"),
                   example=("accession", lambda s: ";".join(s.head(20))))
              .sort_values("n_proteins", ascending=False).reset_index())
    freq["topic"] = freq.peptide.map(topics["topic_of"])
    freq.to_csv(CFG["OUT"]/"peptide_frequencies.csv", index=False)

def first_meta(row): return acc2meta.get(row.example.split(";")[0], {})

log("top 20 peptides by protein count:")
for r in freq.head(20).itertuples():
    m = first_meta(r)
    print(f"  {r.peptide:<40} n={r.n_proteins:>5} topic={r.topic} "
          f"{m.get('label','?')} {m.get('description','')[:42]}")

# --- RuBisCO share, computed rather than eyeballed ------------------------
RBC = re.compile(r"[Rr]ibulose.*carboxylase")
def rbc_share(k):
    d = [first_meta(r).get("description","") for r in freq.head(k).itertuples()]
    return sum(bool(RBC.search(x)) for x in d), k
for k in (20, 100):
    n_, k_ = rbc_share(k); log(f"RuBisCO large chain in top {k_}: {n_} / {k_}")

# --- fragment enrichment --------------------------------------------------
frag = {a for a,v in acc2meta.items() if v["is_fragment"]}
share200 = np.mean([np.mean([a in frag for a in r.example.split(";")])
                    for r in freq.head(200).itertuples()])
base = df.is_fragment.mean()
log(f"top-200 source proteins fragmentary: {share200:.1%} vs corpus {base:.1%} "
    f"({share200/base:.1f}x enrichment)")

# --- topic purity ---------------------------------------------------------
tt = freq.dropna(subset=["topic"]).copy()
tt["cls"] = tt.example.str.split(";").str[0].map(lambda a: acc2meta.get(a,{}).get("label"))
tt = tt.dropna(subset=["cls"])
purity = tt.groupby("topic")["cls"].agg(lambda s: s.value_counts(normalize=True).iloc[0])
purity.to_csv(CFG["OUT"]/"topic_purity.csv")
log(f"topics >90% one GO class: {(purity>0.9).mean():.1%} | median {purity.median():.3f} "
    f"| min {purity.min():.3f} | n={len(purity):,}")

ARTEFACTS = dict(rubisco_top20=rbc_share(20)[0], rubisco_top100=rbc_share(100)[0],
                 frag_top200=float(share200), frag_corpus=float(base),
                 purity_gt90=float((purity>0.9).mean()), purity_median=float(purity.median()),
                 purity_min=float(purity.min()), n_topics=int(len(purity)))
json.dump(ARTEFACTS, open(CFG["OUT"]/"artefacts.json","w"), indent=2)

## 11. ESM-2 upper bound

Establishes that the task is solvable across this split, so any failure belongs to the
tokenization rather than to the problem.

**Disclosed in Methods:** ESM-2 was pretrained on UniRef, which contains these sequences. This is
an optimistic bound, not a like-for-like arm. It only needs to establish a ceiling.

Requires a GPU runtime. Embeddings checkpoint every 500 sequences, so a timeout costs one chunk.

In [ ]:
!pip install -q transformers
import torch
from transformers import AutoTokenizer, AutoModel

L = df.sequence.str.len()
log(f"{(L>CFG['ESM_MAXLEN']).sum():,}/{len(df):,} sequences over {CFG['ESM_MAXLEN']} aa "
    f"({(L>CFG['ESM_MAXLEN']).mean():.1%}) - truncated")
print(df.assign(L=L).groupby("label").L.apply(lambda s:(s>CFG["ESM_MAXLEN"]).mean()).round(3).to_string())

CK = CFG["OUT"]/"esm2"; CK.mkdir(exist_ok=True)
dev = "cuda" if torch.cuda.is_available() else "cpu"
assert dev == "cuda", "Runtime > Change runtime type > GPU, then re-run."
tok = AutoTokenizer.from_pretrained(CFG["ESM_MODEL"])
esm = AutoModel.from_pretrained(CFG["ESM_MODEL"], dtype=torch.float16).to(dev).eval()

order = df.sequence.str.len().sort_values().index.to_numpy()   # length-sorted: less padding
for start in range(0, len(order), CFG["ESM_CHUNK"]):
    out = CK/f"emb_{start:06d}.npz"
    if out.exists(): continue
    ix = order[start:start+CFG["ESM_CHUNK"]]
    seqs = [df.sequence.iloc[i][:CFG["ESM_MAXLEN"]] for i in ix]
    vecs = []
    with torch.no_grad():
        for j in range(0, len(seqs), CFG["ESM_BATCH"]):
            b = tok(seqs[j:j+CFG["ESM_BATCH"]], return_tensors="pt", padding=True,
                    truncation=True, max_length=CFG["ESM_MAXLEN"]+2)
            b = {k: v.to(dev) for k,v in b.items()}
            h = esm(**b).last_hidden_state
            m = b["attention_mask"].clone(); m[:,0] = 0
            m[torch.arange(len(m)), m.sum(1)] = 0
            m = m.unsqueeze(-1).half()
            vecs.append(((h*m).sum(1)/m.sum(1)).float().cpu().numpy())
    np.savez_compressed(out, idx=ix, emb=np.vstack(vecs))
    print(f"{start+len(ix):,} / {len(order):,}", flush=True)
log("embedding complete")

In [ ]:
X_esm = np.zeros((len(df), 1280), np.float32); seen = np.zeros(len(df), bool)
for f in sorted(CK.glob("emb_*.npz")):
    z = np.load(f); X_esm[z["idx"]] = z["emb"]; seen[z["idx"]] = True
assert seen.all(), f"{(~seen).sum()} sequences never embedded"
np.save(CFG["OUT"]/"X_esm2.npy", X_esm)

run_arm("esm2_650M", X_esm[~test_mask], X_esm[test_mask])

# per-class report: macro-F1 can be depressed by one small class at argmax
rep = pd.DataFrame(reports["esm2_650M"]).T.loc[list(le.classes_),
        ["precision","recall","f1-score","support"]]
save_table(rep.round(4), "T12_esm2_per_class")
print(); print(rep.round(3).to_string())

# is a low-recall class a threshold effect or a representational one?
worst = rep["recall"].idxmin(); c = list(le.classes_).index(worst)
pr = probas["esm2_650M"]; m = (y_test==c)
rank = int(np.median((pr[m] > pr[m,c][:,None]).sum(1)) + 1)
log(f"lowest-recall class {worst}: mean P(true) {pr[m,c].mean():.3f}, median rank {rank}/6")
log("Rank near 1-2 with high precision indicates a decision-threshold effect on a small class,")
log("not a failure to represent it. Report weighted F1 and AUPR alongside macro-F1.")

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
lr = make_pipeline(StandardScaler(),
                   LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=-1))
lr.fit(X_esm[~test_mask], y_train); p_lr = lr.predict(X_esm[test_mask])
log(f"ESM-2 + logistic regression macro-F1 "
    f"{f1_score(y_test, p_lr, labels=ALL_LABELS, average='macro', zero_division=0):.4f} "
    f"(RF kept as primary for comparability across arms)")

## 12. Homology-transfer baseline

The standard baseline in this literature, and the one v1 and v2 never ran. Search test against
train, transfer the label of the highest-scoring hit.

The permissive E-value is deliberate: it lets the identity distribution of best hits be
characterised rather than truncated. Binning and prediction must use the **same** hit criterion,
or proteins that had a hit will be scored as though they had none.

In [ ]:
HOM = CFG["OUT"]/"homology"; HOM.mkdir(exist_ok=True)
def wf(mask, path):
    with open(path,"w") as fh:
        for a,s in zip(df.accession[mask], df.sequence[mask]): fh.write(f">{a}\n{s}\n")
wf(~test_mask, HOM/"train.fasta"); wf(test_mask, HOM/"test.fasta")

hits_p = HOM/"hits.m8"
if not hits_p.exists():
    tmp = CFG["WORK"]/"hom_tmp"; shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir(parents=True)
    r = subprocess.run(["mmseqs","easy-search", str(HOM/"test.fasta"), str(HOM/"train.fasta"),
        "/content/hits.m8", str(tmp), "-s","7.5","-e","1e5","--max-seqs","300",
        "--threads", str(os.cpu_count() or 4),
        "--format-output","query,target,fident,alnlen,qcov,tcov,evalue,bits"],
        capture_output=True, text=True)
    if r.returncode != 0: print(r.stderr[-2000:]); raise RuntimeError("search failed")
    shutil.copy("/content/hits.m8", hits_p)

hits = pd.read_csv(hits_p, sep="\t", header=None,
                   names=["q","t","fident","alnlen","qcov","tcov","evalue","bits"])
acc2label = dict(zip(df.accession, df.label))
te_acc = df.accession[test_mask].to_numpy()
b = hits.sort_values("bits", ascending=False).drop_duplicates("q")
pm = dict(zip(b.q, b.t.map(acc2label)))
has = np.array([a in pm for a in te_acc])
log(f"test proteins with a local hit: {has.sum():,}/{len(has):,}")

fallback = le.classes_[pd.Series(y_train).mode()[0]]
y_blast = le.transform(np.array([pm.get(a, fallback) for a in te_acc]))
preds["blast_transfer"] = y_blast
reports["blast_transfer"] = classification_report(y_test, y_blast, labels=ALL_LABELS,
                                                  target_names=le.classes_, output_dict=True,
                                                  zero_division=0)
rows.append(dict(arm="blast_transfer", n_features=np.nan,
                 accuracy=accuracy_score(y_test,y_blast),
                 balanced_accuracy=balanced_accuracy_score(y_test,y_blast),
                 macro_f1=f1_score(y_test,y_blast,labels=ALL_LABELS,average="macro",zero_division=0),
                 weighted_f1=f1_score(y_test,y_blast,labels=ALL_LABELS,average="weighted",zero_division=0)))
np.save(CFG["OUT"]/"cm_blast_transfer.npy", confusion_matrix(y_test, y_blast, labels=ALL_LABELS))
log(f"blast_transfer macro-F1 {rows[-1]['macro_f1']:.4f} "
    f"| accuracy where a hit exists {(y_blast[has]==y_test[has]).mean():.4f}")

## 13. Final summary

In [ ]:
summ = pd.DataFrame(rows).sort_values("macro_f1", ascending=False)
save_table(summ.round(4), "T13_summary_all_arms")
json.dump(reports, open(CFG["OUT"]/"per_class_reports.json","w"), indent=2, default=float)
save_ck("preds", (preds, le, y_test))
print(summ.round(4).to_string(index=False))

# stratification refreshed with every arm present
rows_s = []
for name, p in preds.items():
    for g in LABELS_ID:
        m = (grp==g).to_numpy()
        if m.sum() >= 20:
            rows_s.append(dict(arm=name, bin=g, n=int(m.sum()),
                               accuracy=float((p[m]==y_test[m]).mean())))
T9 = pd.DataFrame(rows_s); save_table(T9, "T9_identity_stratified")
print("\naccuracy by identity to nearest training protein (descriptive; see §9 confounds):")
print(T9.pivot(index="bin", columns="arm", values="accuracy").reindex(LABELS_ID).round(3).to_string())

## 14. Figures

Six figures for the manuscript, written to `results/figures/` as PDF (vector) and PNG.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family":"DejaVu Sans","font.size":9,"axes.spines.top":False,
                     "axes.spines.right":False,"axes.linewidth":.8,"figure.dpi":150})
FIG = CFG["OUT"]/"figures"
INK, OX, SL, WA = "#16161A", "#A63D2B", "#5C6672", "#D6D0C2"
def save_fig(fig, name):
    for ext in ("pdf","png"): fig.savefig(FIG/f"{name}.{ext}", bbox_inches="tight", facecolor="white")
    plt.close(fig); log(f"figure -> {name}")

# F1 vocabulary growth and occurrence distribution
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
occ = vc_pep.values
axes[0].loglog(np.arange(1, len(occ)+1), occ, color=OX, lw=1.2)
axes[0].set_xlabel("peptide rank"); axes[0].set_ylabel("proteins containing")
axes[0].set_title(f"Tryptic peptide occurrence\n{len(occ):,} unique, "
                  f"{(occ==1).mean():.1%} singletons", fontsize=9)
axes[0].axhline(1, color=SL, lw=.6, ls=":")
sizes_ = [VOCAB_STATS["n_kmer_unique"], len(occ)]
axes[1].bar(["3-mer","tryptic peptide"], sizes_, color=[SL, OX], width=.55)
axes[1].set_yscale("log"); axes[1].set_ylabel("unique tokens")
for i,v in enumerate(sizes_): axes[1].text(i, v*1.15, f"{v:,}", ha="center", fontsize=8)
axes[1].set_title("Vocabulary size", fontsize=9)
save_fig(fig, "F1_vocabulary")

# F2 OOV and zero-vector
fig, ax = plt.subplots(figsize=(5.2, 3.0))
arms_ = ["kmer","peptide"]
oov_ = [extra[f"{a}_oov"] for a in arms_]; zero_ = [extra[f"{a}_zero"]/test_mask.sum() for a in arms_]
x = np.arange(2); w = .35
ax.bar(x-w/2, oov_, w, color=SL, label="token OOV")
ax.bar(x+w/2, zero_, w, color=OX, label="proteins with zero coverage")
ax.set_xticks(x); ax.set_xticklabels(["3-mer","tryptic peptide"]); ax.set_ylim(0,1)
ax.set_ylabel("fraction of test set"); ax.legend(frameon=False, fontsize=8)
for i,(a,bv) in enumerate(zip(oov_, zero_)):
    ax.text(i-w/2, a+.02, f"{a:.1%}", ha="center", fontsize=8)
    ax.text(i+w/2, bv+.02, f"{bv:.1%}", ha="center", fontsize=8)
save_fig(fig, "F2_out_of_vocabulary")

# F3 two populations
fig, ax = plt.subplots(figsize=(6.4, 3.4))
yy = np.arange(len(T6))[::-1]
ax.barh(yy+.18, T6.acc_covered.values, .34, color=INK, label="covered")
ax.barh(yy-.18, T6.acc_zero.fillna(0).values, .34, color=OX, label="zero-vector")
ax.set_yticks(yy); ax.set_yticklabels(T6.index, fontsize=8)
ax.set_xlim(0,1.05); ax.set_xlabel("accuracy"); ax.legend(frameon=False, fontsize=8, loc="lower right")
for i,(cv,zv,nz) in enumerate(zip(T6.acc_covered, T6.acc_zero.fillna(0), T6.n_zero)):
    ax.text(cv+.01, yy[i]+.18, f"{cv:.3f}", va="center", fontsize=7.5, color=INK)
    ax.text(zv+.01, yy[i]-.18, f"{zv:.1f}  (n={nz})", va="center", fontsize=7.5, color=OX)
ax.set_title("Peptide arm: accuracy conditioned on vocabulary coverage", fontsize=9)
save_fig(fig, "F3_two_populations")

# F4 arm comparison
fig, ax = plt.subplots(figsize=(6.4, 3.6))
S = summ.dropna(subset=["macro_f1"]).sort_values("macro_f1")
cols = [SL if a in ("blast_transfer","esm2_650M") else
        (WA if a.startswith("baseline") else OX) for a in S.arm]
ax.barh(np.arange(len(S)), S.macro_f1, color=cols, height=.6)
ax.set_yticks(np.arange(len(S))); ax.set_yticklabels(S.arm, fontsize=8)
ax.set_xlabel("macro-F1"); ax.set_xlim(0,1.05)
for i,v in enumerate(S.macro_f1): ax.text(v+.01, i, f"{v:.4f}", va="center", fontsize=7.5)
save_fig(fig, "F4_arm_comparison")

# F5 confusion matrices (only arms that have been run)
want = [n for n in ["peptide","kmer","esm2_650M","blast_transfer"]
        if (CFG["OUT"]/f"cm_{n}.npy").exists()]
if want:
    nc = len(le.classes_)
    fig, axes = plt.subplots(1, len(want), figsize=(3.4*len(want), 3.4), squeeze=False)
    for ax_, name in zip(axes[0], want):
        cm = np.load(CFG["OUT"]/f"cm_{name}.npy").astype(float)
        cmn = cm/np.maximum(cm.sum(1, keepdims=True), 1)
        ax_.imshow(cmn, cmap="Greys", vmin=0, vmax=1)
        ax_.set_xticks(range(nc)); ax_.set_yticks(range(nc))
        ax_.set_xticklabels([c[-4:] for c in le.classes_], fontsize=7, rotation=90)
        ax_.set_yticklabels([c[-4:] for c in le.classes_], fontsize=7)
        ax_.set_title(name, fontsize=9); ax_.set_xlabel("predicted", fontsize=8)
    axes[0][0].set_ylabel("true", fontsize=8)
    save_fig(fig, "F5_confusion_matrices")
else:
    log("F5 skipped: no confusion matrices on disk yet")

# F6 min_count sensitivity
fig, ax = plt.subplots(figsize=(5.2, 3.0))
ax.plot(T8.min_count, T8.token_oov, "o-", color=OX, label="token OOV")
ax.plot(T8.min_count, T8.zero_pct, "s-", color=SL, label="proteins with zero coverage")
ax.plot(T8.min_count, T8.macro_f1, "^-", color=INK, label="macro-F1")
ax.set_xticks(T8.min_count); ax.set_xlabel("Word2Vec min_count"); ax.set_ylim(0,1)
ax.legend(frameon=False, fontsize=8)
save_fig(fig, "F6_min_count_sensitivity")

### 14b. Journal-style figures

Figures 1–3 and Figure S1 of the manuscript. Run after the figure cell above.

In [ ]:
# ---------------------------------------------------------------------------
# 14b. Journal-style manuscript figures (PDF + 300-dpi PNG written to FIG).
#   Fig1_vocabulary_recurrence, Fig2_coverage_split, Fig3_confusion_matrices (main text)
#   FigS1_min_count (supplement). Confusion matrices are read from cm_<arm>.npy.
# No titles are drawn inside the figures: the manuscript captions carry them.
# ---------------------------------------------------------------------------
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

PEP, KMER, INK, GREY = "#D55E00", "#0072B2", "#222222", "#8A8A8A"   # Okabe-Ito, colour-blind safe
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 8, "axes.labelsize": 8,
                     "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "legend.fontsize": 7.5,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.linewidth": .7, "figure.dpi": 150})


def panel(ax, letter):
    ax.text(-0.02, 1.08, letter, transform=ax.transAxes, fontsize=10, fontweight="bold",
            va="bottom", ha="right")


def save_ms(fig, name):
    for ext, kw in (("pdf", {}), ("png", {"dpi": 300})):
        fig.savefig(FIG / f"{name}.{ext}", bbox_inches="tight", facecolor="white", **kw)
    plt.close(fig)
    log(f"manuscript figure -> {name}")


def pct(v):                                   # honest labels for very small values
    return f"{v*100:.3f}%" if 0 < v < 0.001 else f"{v*100:.1f}%"


# ---- Figure 1: the peptide vocabulary does not recur ------------------------
occ = np.asarray(vc_pep.values)
n_pep, n_kmer = len(occ), VOCAB_STATS["n_kmer_unique"]
single = float((occ == 1).mean())
fig, ax = plt.subplots(1, 3, figsize=(7.0, 2.6), gridspec_kw={"width_ratios": [1.25, 1, 1.15]})
ax[0].loglog(np.arange(1, n_pep + 1), occ, color=PEP, lw=1.3)
ax[0].axhline(1, color=GREY, lw=.6, ls=":")
ax[0].set_xlabel("Peptide rank"); ax[0].set_ylabel("Proteins containing\nthe peptide")
ax[0].text(0.97, 0.95, f"{single:.1%} of {n_pep:,}\npeptides occur once", transform=ax[0].transAxes,
           ha="right", va="top", fontsize=7.5, color=INK)
panel(ax[0], "A")
ax[1].set_yscale("log")
for x, v, c in ((0, n_kmer, KMER), (1, n_pep, PEP)):
    ax[1].vlines(x, 1, v, color=c, lw=2.2)
    ax[1].plot(x, v, "o", color=c, ms=6)
    ax[1].text(x + .14, v, f"{v:,}", ha="left", va="center", fontsize=7.5)
ax[1].set_xticks([0, 1]); ax[1].set_xticklabels(["3-mer", "Tryptic\npeptide"])
ax[1].set_xlim(-.5, 2.0); ax[1].set_ylim(1e3, 2e6); ax[1].set_ylabel("Unique tokens (log scale)")
panel(ax[1], "B")
oov = [extra["kmer_oov"], extra["peptide_oov"]]
zero = [extra["kmer_zero"] / test_mask.sum(), extra["peptide_zero"] / test_mask.sum()]
xs, w = np.arange(2), .36
ax[2].bar(xs - w / 2, oov, w, color=INK, label="Test tokens unseen in training")
ax[2].bar(xs + w / 2, zero, w, color=PEP, label="Test proteins with no\npeptide seen in training")
for i in range(2):
    # stagger the two labels when both bars are tiny, so they do not collide
    dy2 = .09 if (oov[i] < .05 and zero[i] < .05) else .02
    ax[2].text(xs[i] - w / 2, oov[i] + .02, pct(oov[i]), ha="center", fontsize=7)
    ax[2].text(xs[i] + w / 2, zero[i] + dy2, pct(zero[i]), ha="center", fontsize=7)
ax[2].set_xticks(xs); ax[2].set_xticklabels(["3-mer", "Tryptic\npeptide"])
ax[2].set_ylim(0, 1.0); ax[2].set_ylabel("Fraction of test set")
ax[2].legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.30), handlelength=1)
panel(ax[2], "C")
fig.tight_layout(w_pad=1.4)
save_ms(fig, "Fig1_vocabulary_recurrence")

# ---- Figure 2: two populations (coverage split) -----------------------------
D = T6.copy()
ypos = np.arange(len(D))[::-1]
fig, ax = plt.subplots(figsize=(4.6, 3.0))
ax.barh(ypos + .19, D.acc_covered.values, .34, color=INK, label="Contains a peptide seen in training")
ax.barh(ypos - .19, D.acc_zero.fillna(0).values, .34, color=PEP, label="No peptide seen in training")
ax.set_yticks(ypos); ax.set_yticklabels(D.index, fontsize=7.5)
ax.set_xlim(0, 1.22); ax.set_xticks([0, .25, .5, .75, 1]); ax.set_xlabel("Accuracy")
for i, (cv, zv, nz, n) in enumerate(zip(D.acc_covered, D.acc_zero.fillna(0), D.n_zero, D.n)):
    ax.text(cv + .015, ypos[i] + .19, f"{cv:.3f}  (n = {n - nz})", va="center", fontsize=6.8, color=INK)
    ax.text(zv + .015, ypos[i] - .19, f"{zv:.3f}  (n = {nz})", va="center", fontsize=6.8, color=PEP)
ax.legend(frameon=False, loc="lower center", bbox_to_anchor=(0.45, 1.0), ncol=1, handlelength=1)
fig.tight_layout()
save_ms(fig, "Fig2_coverage_split")

# ---- Figure 3: confusion matrices -------------------------------------------
names = {"peptide": "Tryptic peptide", "kmer": "3-mer", "esm2_650M": "ESM-2", "blast_transfer": "Homology transfer"}
want = [a for a in names if (CFG["OUT"] / f"cm_{a}.npy").exists()]
if want:
    nc = len(le.classes_)
    fig, axes = plt.subplots(1, len(want), figsize=(7.0, 2.5), squeeze=False)
    for k, (ax_, a) in enumerate(zip(axes[0], want)):
        cm = np.load(CFG["OUT"] / f"cm_{a}.npy").astype(float)
        cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
        im = ax_.imshow(cmn, cmap="Greys", vmin=0, vmax=1)
        ax_.set_xticks(range(nc)); ax_.set_yticks(range(nc))
        ax_.set_xticklabels([c[-4:] for c in le.classes_], rotation=90, fontsize=6.5)
        ax_.set_yticklabels([c[-4:] for c in le.classes_] if k == 0 else [], fontsize=6.5)
        ax_.set_title(names[a], fontsize=8); ax_.set_xlabel("Predicted class")
        if k == 0: ax_.set_ylabel("True class")
        panel(ax_, "ABCD"[k])
        for s in ("top", "right"): ax_.spines[s].set_visible(True)
    fig.tight_layout(rect=(0, 0, .93, 1))
    cax = fig.add_axes([.945, .2, .012, .6]); cb = fig.colorbar(im, cax=cax)
    cb.set_label("Fraction of true class", fontsize=7); cb.ax.tick_params(labelsize=6.5)
    save_ms(fig, "Fig3_confusion_matrices")
else:
    log("Fig3 skipped: no cm_*.npy files found")

# ---- Figure S1: min_count sensitivity ---------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(5.6, 2.4))
ax[0].plot(T8.min_count, T8.token_oov, "o-", color=INK, label="Test tokens unseen in training")
ax[0].plot(T8.min_count, T8.zero_pct, "s--", color=PEP, label="Test proteins with no peptide\nseen in training")
ax[0].set_ylim(0, 1); ax[0].set_ylabel("Fraction of test set")
ax[0].legend(frameon=False, loc="center", bbox_to_anchor=(0.52, 0.60), fontsize=6.8)
ax[1].plot(T8.min_count, T8.macro_f1, "^-", color=KMER)
ax[1].set_ylim(0, 1); ax[1].set_ylabel("Macro-F1")
for a_, L in zip(ax, "AB"):
    a_.set_xticks(list(T8.min_count)); a_.set_xlabel("Word2Vec minimum count"); panel(a_, L)
fig.tight_layout(w_pad=1.6)
save_ms(fig, "FigS1_min_count")

## 15. Run manifest

Captures everything Methods needs, so the manuscript never quotes a value that is not on disk.

In [ ]:
import sklearn, gensim, scipy
try:
    import transformers, torch
    tv, thv = transformers.__version__, torch.__version__
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
except Exception:
    tv = thv = gpu = "not loaded"

MANIFEST = {
  "uniprot": json.load(open(CFG["OUT"]/"download_meta.json")),
  "corpus": {"n": int(len(df)), "database_mix": df.database.value_counts().to_dict(),
             "fragment_frac": float(df.is_fragment.mean()),
             "class_counts": df.label.value_counts().sort_index().to_dict(),
             "trunc_1022_frac": float((df.sequence.str.len()>1022).mean())},
  "split": {"strategy": "cluster_disjoint", "fold_used": int(FOLD_USED),
            "n_clusters": int(n_clusters), "redundancy": float(len(df)/n_clusters),
            "n_components": int(ncomp),
            "largest_component_frac": float(pd.Series(comp).value_counts().iloc[0]/len(df)),
            "n_train": int((~test_mask).sum()), "n_test": int(test_mask.sum()),
            "identity_ge_30": float((gi>=0.30).mean()), "identity_median": float(np.median(gi))},
  "vocabulary": VOCAB_STATS,
  "oov": {a: float(extra[f"{a}_oov"]) for a in ("kmer","peptide")},
  "zero_vector": {a: int(extra[f"{a}_zero"]) for a in ("kmer","peptide")},
  "n_topics": int(topics["k"]),
  "artefacts": json.load(open(CFG["OUT"]/"artefacts.json")),
  "software": {"python": sys.version.split()[0], "numpy": np.__version__,
               "pandas": pd.__version__, "scipy": scipy.__version__,
               "scikit_learn": sklearn.__version__, "gensim": gensim.__version__,
               "transformers": tv, "torch": thv, "mmseqs2": MMSEQS_VERSION, "gpu": gpu},
  "config": {k: str(v) for k,v in CFG.items()},
}
json.dump(MANIFEST, open(CFG["OUT"]/"run_manifest.json","w"), indent=2)
print(json.dumps({k:v for k,v in MANIFEST.items() if k != "config"}, indent=2)[:2600])
log(f"manifest -> {CFG['OUT']/'run_manifest.json'}")
log(f"tables -> {CFG['OUT']/'tables'} | figures -> {CFG['OUT']/'figures'}")

## 16. Corpus accession list and split assignment

Writes accession, class, database, fragment flag and train/test assignment. No sequences are included.

In [ ]:
# Accession list and split assignment for the corpus. Contains NO sequences, so it can be
# shared; it lets others retrieve exactly these proteins from UniProt by accession.
acc_tab = df[["accession", "label", "database", "is_fragment"]].copy()
acc_tab["split"] = np.where(test_mask, "test", "train")
acc_tab.to_csv(CFG["OUT"]/"tables"/"corpus_accessions_and_split.tsv", sep="\t", index=False)
log(f"accession list -> {CFG['OUT']/'tables'/'corpus_accessions_and_split.tsv'} "
    f"({len(acc_tab):,} proteins; {int(test_mask.sum()):,} test)")